# COMP3122 — Week 4 Demo Notebook
### Chapter 4: Pandas — Series, DataFrames, and Loading Data

**Applied Data Science**

This notebook is the companion to the Week 4 lecture. It follows the same six blocks as the
slides, in the same order, on the same four cities.

**In class, your instructor drives this and you watch.** You are not expected to type along.
**Afterwards it is yours** — run it top to bottom at your own pace. Every cell depends on the
ones above it, in particular on `weather`, which is built once in Block 2.

Cells marked **LIVE DEMO** are the ones the lecture stops for. Three of them deliberately
raise errors — that is the point, so read the message rather than fixing it.

The `# Your turn` cells are for after the lecture. Try each one before you open the answer
cell underneath it.

---
## Setup

Two imports. `pd` and `np` are conventions, not preferences — every tutorial, book and Stack
Overflow answer uses them.

In [1]:
import pandas as pd          # explain: the standard short name for the Pandas library
import numpy as np           # explain: still needed — a DataFrame is built on NumPy

print("pandas:", pd.__version__)    # explain: this course pins 2.2.2 — flag it if you see another
print("numpy :", np.__version__)

pandas: 3.0.6
numpy : 2.5.3


---
## Block 1 — From arrays to tables  ·  Book §4.1–4.2

Here is the array you already know how to build. Four cities, seven days, a warmer week than
Chapter 2's — same shape, same layout.

In [2]:
# explain: daily highs in °C for 4 cities across 7 days
temps = np.array([
    [22, 24, 25, 27, 26, 25, 23],   # Toronto [0]
    [30, 31, 33, 34, 33, 32, 30],   # Cairo   [1]
    [18, 19, 19, 20, 21, 20, 19],   # Tokyo   [2]
    [28, 27, 26, 25, 26, 27, 28],   # Sydney  [3]
])

print(temps)
print(temps.shape)   # explain: (4, 7) — but WHICH row is Tokyo? The array cannot say.

[[22 24 25 27 26 25 23]
 [30 31 33 34 33 32 30]
 [18 19 19 20 21 20 19]
 [28 27 26 25 26 27 28]]
(4, 7)


The labels live in the comments and in your head. The array does not know them. That is the
whole problem Pandas exists to fix.

### The Series — a labelled 1D array

In [3]:
# explain: Toronto's seven daily highs, each one tagged with a day name
toronto = pd.Series(
    data=[22, 24, 25, 27, 26, 25, 23],
    index=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    name="Toronto"
)
print(toronto)

Mon    22
Tue    24
Wed    25
Thu    27
Fri    26
Sat    25
Sun    23
Name: Toronto, dtype: int64


Three things in that output. Every value has a label on the left — that strip of labels is
the **index**. The Series has a `name`. And the **dtype** is `int64`, exactly as in NumPy.

> ⚠️ **"Index" is the most overloaded word in this chapter.** In plain Python, *index* means
> position. In Pandas, *index* means the row **labels**. Keep them apart or Block 3 will hurt.

### 🔴 LIVE DEMO 1 — what happens without `index=`

Run the cell below, then compare it with the Series above. Which one would you rather read
six months from now?

In [4]:
# explain: the SAME seven values, with no index argument
no_labels = pd.Series([22, 24, 25, 27, 26, 25, 23])
print(no_labels)   # explain: Pandas fills in 0..6 — back to positions, back to guessing

0    22
1    24
2    25
3    27
4    26
5    25
6    23
dtype: int64


### By label, by position — and the slicing trap

In [5]:
print(toronto["Wed"])     # explain: by LABEL → 25
print(toronto.iloc[2])    # explain: by POSITION → 25 (same value, different question)

25
25


In [6]:
print(toronto["Mon":"Wed"])   # explain: label slice → Mon, Tue, Wed — 3 values
print(toronto.iloc[0:3])      # explain: position slice → also 3 values here

Mon    22
Tue    24
Wed    25
Name: Toronto, dtype: int64
Mon    22
Tue    24
Wed    25
Name: Toronto, dtype: int64


Both returned three values. **That is a coincidence of where they start, not a rule.**

Label slicing **includes** the upper end. Position slicing **excludes** it, like a Python
list. Read the rule off the ends, never off the count.

### Arithmetic and aggregations keep the labels

In [7]:
toronto_f = toronto * 9 / 5 + 32   # explain: convert every value to Fahrenheit at once
print(toronto_f)                   # explain: every value converted — every label still there

Mon    71.6
Tue    75.2
Wed    77.0
Thu    80.6
Fri    78.8
Sat    77.0
Sun    73.4
Name: Toronto, dtype: float64


In [ ]:
print(toronto.mean())              # explain: the week's average → 24.571428...
print(round(toronto.mean(), 2))    # explain: → 24.57
print(round(toronto.std(), 2))     # explain: → 1.72 — SAMPLE std (ddof=1), Pandas' default
print(round(toronto.std(ddof=0), 2))  # explain: → 1.59 — the NumPy default; same data, different formula
print(toronto.max())               # explain: the hottest day → 27

> **Quick Check 4.2.** Build a Series called `cairo` for `[30, 31, 33, 34, 33, 32, 30]` with
> the same `Mon`–`Sun` index. What do `cairo["Wed"]`, `cairo.iloc[2]` and
> `cairo.mean().round(2)` return?

In [ ]:
# Your turn. Write it here, then run the cell below to check.


In [ ]:
# explain: the answer — run this after you have tried it yourself
cairo = pd.Series([30, 31, 33, 34, 33, 32, 30],
                  index=["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], name="Cairo")
print(cairo["Wed"])             # explain: → 33
print(cairo.iloc[2])            # explain: → 33
print(cairo.mean().round(2))    # explain: → 31.86  (223 / 7 = 31.857142...)

---
## Block 2 — The DataFrame  ·  Book §4.3

A DataFrame is a table: rows, columns, and a value at every crossing. The easiest way to
build one is from a dictionary — every key becomes a column.

In [ ]:
# explain: weekly highs for four cities, as a labelled table
weather = pd.DataFrame(
    data={
        "Mon": [22, 30, 18, 28],
        "Tue": [24, 31, 19, 27],
        "Wed": [25, 33, 19, 26],
        "Thu": [27, 34, 20, 25],
        "Fri": [26, 33, 21, 26],
        "Sat": [25, 32, 20, 27],
        "Sun": [23, 30, 19, 28],
    },
    index=["Toronto", "Cairo", "Tokyo", "Sydney"]   # explain: the row labels
)
weather   # explain: last line of a cell displays — Jupyter renders this as a real table

### 🔴 LIVE DEMO 2 — the seven inspection tools

Run all seven. **Every time you meet a new dataset, for the rest of your career.** It takes
four seconds and it catches the mistakes that otherwise cost an afternoon.

In [ ]:
print(weather.shape)        # explain: (rows, columns) → (4, 7)

In [ ]:
weather.head(2)             # explain: first 2 rows; the default is 5

In [ ]:
weather.tail(2)             # explain: last 2 rows — catches truncated file endings

In [ ]:
weather.info()              # explain: columns, non-null counts, dtypes — PRINTS, returns nothing

In [ ]:
weather.describe()          # explain: count, mean, std (sample, ddof=1), min, percentiles, max

In [ ]:
print(weather.dtypes)       # explain: the type of each column → all int64

In [ ]:
print(weather.columns)      # explain: the column labels, in order

**Four are methods and need `()`:** `head`, `tail`, `info`, `describe`.
**Three are attributes and take none:** `shape`, `dtypes`, `columns`.

Getting that wrong is the first error most people make. `weather.shape()` is a `TypeError`.

### A column is a Series — one bracket or two?

In [ ]:
wed = weather["Wed"]        # explain: ONE bracket → a Series, indexed by city
print(type(wed))            # explain: <class 'pandas.core.series.Series'>
print(wed.shape)            # explain: → (4,)

In [ ]:
wed_frame = weather[["Wed"]]   # explain: TWO brackets → a DataFrame with one column
print(type(wed_frame))         # explain: <class 'pandas.core.frame.DataFrame'>
print(wed_frame.shape)         # explain: → (4, 1)

Identical numbers, different type. The inner brackets are a plain Python **list** of column
names — so a one-element list gives you back a one-column table. Code downstream that expects
a DataFrame will break on a Series, and the traceback will blame that code, not your brackets.

### Underneath, it is still NumPy

In [ ]:
print(weather.to_numpy())         # explain: the raw 2D array hiding inside
print(weather.to_numpy().shape)   # explain: → (4, 7)
print(weather.index)              # explain: the row labels, as an Index object
print(weather.columns)            # explain: the column labels, as an Index object

A DataFrame is a NumPy array **plus two label sequences**. Which means Chapter 3's `axis`
rule still works — and now the answer comes back labelled.

In [ ]:
print(weather.mean(axis=1).round(2))   # explain: collapse the day COLUMNS → one mean per city
print(weather.mean(axis=0).round(2))   # explain: collapse the city ROWS → one mean per day

> **Quick Check 4.3.** Without running it: what do `weather["Sun"].max()`,
> `weather[["Mon", "Tue"]].shape` and `weather.mean(axis=1)["Tokyo"]` (2 dp) return?

In [ ]:
# Your turn.


In [ ]:
# explain: the answers
print(weather["Sun"].max())                      # explain: → 30 (Cairo)
print(weather[["Mon","Tue"]].shape)              # explain: → (4, 2) — double brackets, so a DataFrame
print(round(weather.mean(axis=1)["Tokyo"], 2))   # explain: → 19.43, asked for BY NAME

---
## Block 3 — `.loc` and `.iloc`  ·  Book §4.4

The most error-prone topic in beginner Pandas. The rule is two lines:

- **`.loc[row_label, column_label]`** — select by **label**
- **`.iloc[row_position, column_position]`** — select by **integer position**

The `i` in `iloc` stands for *integer*. That is the mnemonic.

In [ ]:
print(weather.loc["Tokyo", "Wed"])   # explain: Tokyo's Wednesday, by label → 19
print(weather.iloc[2, 2])            # explain: row 2, column 2, by position → also 19

Same cell, two ways of asking. Read your selection aloud: *"Tokyo, Wednesday"* tells you
whether you meant it. *"Two, two"* tells you nothing.

### 🔴 LIVE DEMO 3 — the two errors you will hit this week

Both of the cells below **raise on purpose**. Read the message, then move on.

In [ ]:
# explain: .loc wants a LABEL, and there is no row labelled 0 — raises KeyError
weather.loc[0]

In [ ]:
# explain: .iloc wants an INTEGER, not a name — raises TypeError
weather.iloc["Toronto"]

They do not fall back to each other. Pandas will not quietly do the other thing.

### Whole rows, whole columns, and lists of them

In [ ]:
print(weather.loc["Cairo"])       # explain: all 7 of Cairo's values, indexed by day
print(weather.loc[:, "Thu"])      # explain: all 4 cities' Thursday, indexed by city
print(weather.iloc[1])            # explain: same as loc["Cairo"] — second row by position

In [ ]:
weather.loc[["Toronto", "Cairo"], ["Mon", "Sun"]]   # explain: a 2 x 2 sub-table

### Slicing — and the one that returns nothing

In [ ]:
print(weather.loc["Toronto":"Tokyo"].shape)   # explain: → (3, 7). Tokyo IS included.
print(weather.iloc[0:3].shape)                # explain: → (3, 7). Position 3 is NOT.

In [ ]:
backward = weather.loc[:, "Wed":"Mon"]   # explain: start at Wed, walk FORWARD, never reach Mon
print(backward.shape)                    # explain: → (4, 0). Empty. And no error at all.

> ⚠️ An empty result with **no error** is the worst kind of bug, because your code keeps
> running. If a slice surprises you with nothing, check the **order** first.

If you need a different order, pass an explicit list:

In [ ]:
weather.loc[:, ["Wed", "Tue", "Mon"]]   # explain: three columns, in the order you asked for

> **Quick Check 4.4.** Write Sydney's Friday high with `.loc`, then with `.iloc`. Which one
> still works if the rows get sorted into a different order, and why?

In [ ]:
# Your turn.


In [ ]:
# explain: the answers
print(weather.loc["Sydney", "Fri"])   # explain: → 26
print(weather.iloc[3, 4])             # explain: → 26 (row 3 = Sydney, column 4 = Fri)

# explain: now sort the table and ask both questions again
shuffled = weather.sort_values("Wed")
print(shuffled.loc["Sydney", "Fri"])  # explain: still 26 — labels travel with the data
print(shuffled.iloc[3, 4])            # explain: 33 — position 3 is Cairo now. Silently WRONG.

**Silently wrong is worse than crashed.** A crash tells you where to look. A plausible wrong
number can travel all the way to a client.

---
## Block 4 — Filtering and sorting  ·  Book §4.5

Same two-step as NumPy masks. Build the mask, then use it.

In [ ]:
print(weather["Wed"] > 25)   # explain: one True/False per city — Toronto False (25 is not > 25)

In [ ]:
hot_wed = weather[weather["Wed"] > 25]   # explain: pass the mask back in
hot_wed                                  # explain: Cairo and Sydney — shape (2, 7)

The pattern is always `dataframe[boolean_series]`. In this row-filtering pattern the mask
removes **rows**; the columns stay unchanged.

Now compare that with Chapter 3 — **the mechanism, not the numbers.** Chapter 3 used a cooler
week; the `temps` above is this chapter's warmer one. What carries over is the behaviour:
there the mask had one value per **cell**, the matches were scattered, and NumPy could only
hand back a flat line.

In [ ]:
print(temps[temps > 25])              # explain: cell mask → flat 1D, 15 values, no row info
print(temps[temps[:, 2] > 25].shape)  # explain: ROW mask → (2, 7). NumPy keeps rows too.

So the difference is not the shape — it is the **names**. NumPy hands you two anonymous rows;
Pandas hands you Cairo and Sydney with Mon–Sun still across the top.

### Combining conditions — `&`, `|`, `~`

In [ ]:
print(weather[(weather["Wed"] > 25) & (weather["Sun"] < 30)].index.tolist())  # AND → Sydney
print(weather[(weather["Mon"] < 20) | (weather["Sat"] > 30)].index.tolist())  # OR  → Cairo, Tokyo
print(weather[~(weather["Fri"] > 30)].index.tolist())                         # NOT → all but Cairo

### 🔴 LIVE DEMO 4 — why `and` does not work

This one raises on purpose too. It is one of the three error messages every Pandas user learns
to recognise on sight.

In [ ]:
# explain: Python's `and` asks "is this thing true?" — a Series of four values has no answer
weather["Wed"] > 25 and weather["Sun"] < 30

Two rules, both mandatory: use `&` `|` `~`, and **wrap every comparison in its own
parentheses**. Without them, Python's precedence binds `&` tighter than `>`, and the error you
get will not mention parentheses.

### Filter, then pick a column

In [ ]:
print(weather[weather["Wed"] > 25]["Wed"])       # explain: works, but chains two lookups
print(weather.loc[weather["Wed"] > 25, "Wed"])   # explain: PREFER THIS — one lookup, clearer

### Sorting

In [ ]:
print(weather.sort_values("Wed").index.tolist())                    # explain: ascending → Tokyo first
print(weather.sort_values("Wed", ascending=False).index.tolist())   # explain: descending → Cairo first
print(weather.index.tolist())                                       # explain: the ORIGINAL is unchanged

`sort_values` returns a **new** DataFrame. Assign it if you want to keep it. And the index
moves with the rows — Tokyo goes to the top and is still called Tokyo.

> **Quick Check 4.5.** One line for the rows where Tuesday ≥ 25 **and** Thursday ≤ 30. How
> many cities match? Then one line for the whole `weather` table sorted by Tuesday, descending.

In [ ]:
# Your turn.


In [ ]:
# explain: the answers
print(weather[(weather["Tue"] >= 25) & (weather["Thu"] <= 30)].index.tolist())
# explain: → ['Sydney'] — ONE row. Toronto fails on Tue (24), Cairo on Thu (34), Tokyo on Tue (19).
print(weather.sort_values("Tue", ascending=False).index.tolist())

---
## Block 5 — Reading and writing CSV files  ·  Book §4.6

Everything so far was typed by hand. Nobody types data by hand.

The cell below writes the five sample CSV files the chapter uses. They also ship with the
chapter — this cell just means the notebook runs anywhere, even on its own.

In [ ]:
from pathlib import Path

# explain: the five sample files, written so this notebook is self-contained
Path("weather.csv").write_text(
    "city,Mon,Tue,Wed,Thu,Fri,Sat,Sun\n"
    "Toronto,22,24,25,27,26,25,23\n"
    "Cairo,30,31,33,34,33,32,30\n"
    "Tokyo,18,19,19,20,21,20,19\n"
    "Sydney,28,27,26,25,26,27,28\n")

Path("euro_format.csv").write_text(     # explain: semicolons, not commas
    "city;temperature;humidity\nParis;18;72\nBerlin;15;80\nMadrid;24;58\nRome;22;65\n")

Path("with_preamble.csv").write_text(   # explain: two junk lines before the real header
    "# Source: values constructed for COMP3122 Chapter 4 (illustrative, not observations)\n"
    "# Note: values are daily highs in degrees Celsius\n"
    "city,Mon,Tue,Wed\nToronto,22,24,25\nCairo,30,31,33\n"
    "Tokyo,18,19,19\nSydney,28,27,26\n")

Path("messy.csv").write_text(           # explain: missing values spelled in a way Pandas will not guess
    "city,Mon,Tue,Wed\nToronto,22,24,25\nCairo,30,unknown,33\n"
    "Tokyo,missing,19,19\nSydney,28,27,unknown\n")

Path("enrollments.csv").write_text(
    "course,section,enrolled,capacity,delivery\n"
    "COMP3122,A,38,40,in-person\nCOMP3122,B,42,40,in-person\n"
    "COMP1011,A,35,40,online\nCOMP1011,B,28,40,online\n"
    "COMP2152,A,40,40,hybrid\nCOMP2152,B,22,40,hybrid\n")

print("five sample files written")

A CSV is a **text file** and nothing more. First line is the header; every other line is one
row. No formatting, no formulas, no sheets — and **no types**. Every value is text until
`read_csv` guesses otherwise. Most trouble in this block is a guess nobody checked.

### 🔴 LIVE DEMO 5 — the one decision worth making consciously

In [ ]:
with_index = pd.read_csv("weather.csv", index_col="city")   # explain: city becomes the row LABELS
without    = pd.read_csv("weather.csv")                     # explain: city stays an ordinary column

print("with index_col   :", with_index.shape, list(with_index.columns))
print("without index_col:", without.shape,    list(without.columns))

Neither is wrong. Seven columns and named rows, or eight columns and numbered rows. Choose
whichever makes the data clearest — **and remember which**, because saving it back has to agree.

### Then run the seven. Yes, again.

In [ ]:
loaded = with_index
print(loaded.shape)
print(loaded.dtypes)
loaded.head()

### Four `read_csv` arguments worth knowing by name

In [ ]:
print(pd.read_csv("euro_format.csv").shape)              # explain: WITHOUT sep → (4, 1). One giant column.
print(pd.read_csv("euro_format.csv", sep=";").shape)     # explain: WITH sep → (4, 3). Correct.

In [ ]:
preamble = pd.read_csv("with_preamble.csv", skiprows=2)  # explain: skip the two comment lines
print(list(preamble.columns))                            # explain: → ['city', 'Mon', 'Tue', 'Wed']
# explain: without skiprows this raises a ParserError — line 1 looks like a 1-column header

In [ ]:
print(pd.read_csv("messy.csv").dtypes)                                     # explain: ALL object — unusable
print(pd.read_csv("messy.csv", na_values=["unknown","missing"]).dtypes)    # explain: float64 — usable

In [ ]:
print(pd.read_csv("weather.csv", usecols=["city", "Wed"]).shape)   # explain: → (4, 2). Two columns only.

Pandas already treats `""`, `NA`, `N/A`, `NaN`, `NULL` and `None` as missing **by default**.
You only need `na_values=` for tokens outside that list, like `"unknown"`.

### A real table — strings and numbers side by side

In [ ]:
enrollments = pd.read_csv("enrollments.csv")
enrollments

In [ ]:
print(enrollments.dtypes)   # explain: 3 object (text) columns, 2 int64 — one table, mixed types

A single NumPy array cannot hold that without coercing everything to one dtype. That is the
second reason Pandas exists, arriving on cue.

In [ ]:
# explain: a column compared against ANOTHER column — over capacity
print(enrollments.loc[enrollments["enrolled"] > enrollments["capacity"], "course"].tolist())

# explain: a string comparison combined with a numeric one
mask = (enrollments["delivery"] == "in-person") & (enrollments["enrolled"] >= 38)
print(enrollments.loc[mask, ["course", "section"]])

print(round(enrollments["enrolled"].mean(), 2))   # explain: → 34.17

### Saving — and the setting that must agree

In [ ]:
enrollments.to_csv("enrollments_clean.csv", index=False)   # explain: default index means nothing — drop it
print(pd.read_csv("enrollments_clean.csv").shape)          # explain: → (6, 5). Correct round trip.

In [ ]:
# explain: THE TRAP — written with index=False, then read with index_col=0
trap = pd.read_csv("enrollments_clean.csv", index_col=0)
print(trap.shape)              # explain: → (6, 4). A column vanished.
print(trap.index.name)         # explain: 'course' — your first DATA column became the row labels.
# explain: no error, no warning. Everything downstream is now off by one.

Two clean pairs. Learn them as pairs:

| index | write | read back |
|---|---|---|
| default `0, 1, 2` | `to_csv(p, index=False)` | `pd.read_csv(p)` |
| meaningful (cities, dates, IDs) | `to_csv(p)` | `pd.read_csv(p, index_col=0)` |

> **Quick Check 4.6.** `weather.dtypes` shows `Wed` as `object`, not `int64`. Name two reasons
> that could happen. What is the first thing you would do to investigate?

In [ ]:
# Your turn.


**Answers.** Any two of: an unrecognised non-numeric value (`"unknown"`, `"25 degrees"`, a
stray letter); a currency symbol like `"$25"`; a different decimal mark, e.g. `"25,5"`.

Note what is *not* on the list: `"N/A"` and `"NULL"`. Pandas handles those by default — they
become `NaN`, which makes the column `float64`, not `object`. Stray spaces are not on the list
either: `read_csv` strips them before inferring, so `" 25 "` still comes in as `int64`.

**First step:** open the file in a plain text editor and read the first ten lines of that
column. Not a Pandas argument — your eyes.

---
## Block 6 — The mindset shift  ·  Book §4.7

> **From positions to labels: a DataFrame is a NumPy array with names.**

Three habits follow, and they are the habits of every working Pandas user:

1. **Inspect before you act.** All seven, every new DataFrame, no exceptions.
2. **Labels first, positions second.** `.loc` is the natural move; `.iloc` is the fallback for
   when you genuinely mean a position.
3. **Keep filters readable.** `df[(c1) & (c2)]` — the parentheses are mandatory, and the same
   shape handles two conditions or twelve.

And one that does not fit a numbered list: **trust the labels.** If you catch yourself
thinking *"I need to remember that row 3 used to be Cairo"* — stop. The DataFrame remembers.

### The seven traps, in one cell

Each line below is a real mistake from §4.7's Common Mistakes. Uncomment one at a time.

In [ ]:
# 1. .loc wants labels, .iloc wants integers
# weather.loc[0]                    # KeyError
# weather.iloc['Toronto']           # TypeError

# 2. .loc slicing includes the upper end; .iloc does not
# weather.loc['Toronto':'Tokyo']    # THREE rows

# 3. one bracket is a Series, two is a DataFrame
# weather['Wed'].shape              # (4,)
# weather[['Wed']].shape            # (4, 1)

# 4. and / or / not do not work on Series — use & | ~ with parentheses

# 5. one stray '$25' turns a whole column to text — always check dtypes after loading

# 6. df.Wed breaks on spaces, digits and method names. df['Wed'] always works.
# weather.count                     # this is a METHOD, not your data

# 7. labels are case-sensitive
# weather['wed']                    # KeyError — check spelling, then capitals, then spaces


---
## Block 7 — A real dataset  ·  the outline's *explore a real dataset*

Everything above was a table typed in for the lesson — clean on purpose. This one is real:
344 penguins measured at Palmer Station, Antarctica, 2007–2009 (Gorman, Williams and Fraser
2014, *PLoS ONE* 9(3): e90081; Palmer Station LTER, released CC0 as `palmerpenguins`).
`penguins.csv` is in this week's bundle; `fetch_week(4)` puts it beside this notebook.
Lab 4 Part 7 asks you to explore it yourself.

In [ ]:
penguins = pd.read_csv("penguins.csv")   # explain: a real file: 344 rows, 7 columns, and some gaps
print(penguins.shape)                    # explain: → (344, 7)
penguins.head()                          # explain: species, island, four measurements, sex

In [ ]:
penguins.info()      # explain: non-null counts differ per column — 342 measurements, 333 sexes: REAL data has holes

In [ ]:
penguins.describe()  # explain: count 342 (not 344) for every measurement — describe skips the NaN values

In [ ]:
# explain: one question a field team would ask: how heavy is a Gentoo, on average?
gentoo = penguins.loc[penguins["species"] == "Gentoo", "body_mass_g"]
print(round(gentoo.mean(), 1))          # explain: → 5076.0 g — about 1.4 kg more than the other two species

Three things that never happened with `weather`: the row count and the measurement count
disagree (missing values), a column of text (`sex`) has *more* gaps than the numbers, and the
answer to a question comes with a caveat — two Gentoo rows have no mass at all. That caveat
is the difference between a table and a dataset.

---
## Before next class

Lecture quiz (deadline on the quiz itself) · finish Lab 04 · read Chapter 5.

Chapter 5 picks this DataFrame up and **looks at it** — line plots, histograms, box plots, and
a heatmap of this same `weather` table, with Matplotlib and Seaborn. Its mindset is one
sentence: *always look at the data before you model it.*

**Nothing this week was harder than broadcasting.** What this week had was volume — a lot of
small moves, each one easy, none of which stick by being read. Find a CSV of your own before
the lab — a bank export, a game stats file, anything — and run the six moves on it: load,
inspect, select, filter, sort, save. Twenty minutes on your own data beats two hours on mine.